# 06 model inference

**Owner:** Denis. Original notebook cells: 50.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
import mlflow
import joblib
import numpy as np
import pandas as pd

# Versioned model information from the verified model handoff.
import json
from pathlib import Path
with Path("/Volumes/workspace/default/eye_style_team2/models/mlp_v1/verified_run.json").open(encoding="utf-8") as f:
    verified_model = json.load(f)
MODEL_VERSION = verified_model["model_version"]
MODEL_URI = verified_model["model_uri"]
THRESHOLD = float(verified_model["threshold"])
BASELINE_PATH = verified_model["preprocessor_path"]

# Load the saved artifacts
scoring_model = mlflow.tensorflow.load_model(MODEL_URI)
scoring_pipeline = joblib.load(BASELINE_PATH)

scoring_preprocessor = scoring_pipeline.named_steps["preprocessor"]

REQUIRED_FEATURES = [
    "Age",
    "LeadTime_Days",
    "Scholarship",
    "Hipertension",
    "Diabetes",
    "Alcoholism",
    "Handcap",
    "Gender",
    "Neighbourhood",
    "Appointment_Weekday"
]

def predict_no_show(appointment):
    """
    Predict no-show risk for one appointment.

    The input must contain the 10 required
    booking-time features.
    """

    missing = [
        feature for feature in REQUIRED_FEATURES
        if feature not in appointment
    ]

    if missing:
        raise ValueError(f"Missing features: {missing}")

    record = pd.DataFrame([appointment])[REQUIRED_FEATURES]

    if record.isna().any().any():
        raise ValueError("Appointment contains missing values.")

    if not 0 <= float(record["Age"].iloc[0]) <= 115:
        raise ValueError("Age is outside the validated range.")

    if float(record["LeadTime_Days"].iloc[0]) < 0:
        raise ValueError("Booking lead time cannot be negative.")

    # Apply the fitted preprocessing from the baseline
    processed = scoring_preprocessor.transform(record)

    if hasattr(processed, "toarray"):
        processed = processed.toarray()

    processed = np.asarray(processed, dtype=np.float32)

    # Generate neural-network prediction
    probability = float(
        scoring_model.predict(processed, verbose=0)[0][0]
    )

    return {
        "model_version": MODEL_VERSION,
        "no_show_score": round(probability, 4),
        "follow_up_flag": bool(probability >= THRESHOLD),
        "threshold": THRESHOLD,
        "prediction_point": "booking"
    }

# Test with an existing validation appointment
# Use a deterministic synthetic booking-time fixture if no training session is active.
example = {"Age": 42, "LeadTime_Days": 7, "Scholarship": 0, "Hipertension": 0, "Diabetes": 0, "Alcoholism": 0, "Handcap": 0, "Gender": "F", "Neighbourhood": "JARDIM CAMBURI", "Appointment_Weekday": "Monday"}

result = predict_no_show(example)

print("=== SCORING FUNCTION TEST ===")
print(result)